<span style="color:#9A1A24;text-align:center">

# TÓPICOS DE MATEMÁTICA PARA SISTEMA AUTÓNOMOS

<div style="text-align: right"><i>  </i></div>

***
<center><h1> TRABALHO LETIVO - ENTREGA 2 </h1></center>

***
<span style="color:#000000;text-align:LEFT">

**Nota 1**: Todos elementos do grupo são responsáveis por esta entrega<br>
**Nota 2**: Todos elementos do grupo tomaram conhecimento das regras estabelecidas para o funcionamento da u.c. no [**Moodle do ISEP**](https://moodle.isep.ipp.pt)<br>

***




## GRUPO
> N: 1230758  Nome: Cláudio Pereira  \
> N: 1231133  Nome: Diogo Ferreira  

# Conteúdo


- [1. Desenvolvimento Dashboard de visualização](#1.-Desenvolvimento-Dashboard-de-visualização)     
- [2. Aspeto visual da dashboard desenvolvida](#2.-Aspeto-visual-da-dashboard-desenvolvida)     
- [3. Conclusões e Trabalho Futuro](#3.-Conclusões-e-Trabalho-Futuro)



## 1. Análise dos requisitos

Numa primeira etapa do trabalho foram selecionados alguns requisitos com a função de guiar o seu desenvolvimento.
Os requisitos estipulados na fase anterior foram os seguintes:

* Visualização das trajetórias numa vista global, com todos os robôs no mesmo mapa;
* Visualização das trajetórias numa vista local, com a possibilidade de alterar a missão;
* Visualização de dados como a profundidade, a distância percorrida e a velocidade;
* Possibilidade de aumentar a velocidade de reprodução da missão.

Para os cumprir foram selecionadas previamente bibliotecas que facilitaram o processo. O **Dash** permitiu criar a interface gráfica e a interação com o utilizador. O **pandas** permitiu extrair os dados necessários dos ficheiros de log em formato CSV. O **NumPy** permitiu executar os cálculos matemáticos e algébricos, nomeadamente as matrizes de rotação e a conversão de coordenadas geodésicas para NED. A vista 3D, o radar e os gráficos de análise de dados (profundidade e velocidade) foram realizados com o **Plotly**. O mapa 2D, com OpenStreetMap e OpenSeaMap, foi feito com a biblioteca **dash-leaflet**. Por fim, o **PyYAML** permitiu ler o ficheiro de configuração (`config.yaml`).

## 2. Desenvolvimento Dashboard de visualização

No desenvolvimento da dashboard de visualização de rotas realizadas por três robots underwater, o repositório foi dividido de maneira a separar o desenvolvimento da interface do utilizador (layout) e da lógica de funcionamento (callbacks), de maneira a facilitar tarefas de manutenção e leitura do código. 
Assim, existem os seguintes ficheiros python: config.py onde se encontram centralizadas todas as constantes, leituras do ficheiro de configuração (config.yaml), caminhos para a pasta de logs fornecidos, as cores associadas a cada robô, e ainda as funções auxiliares de mapeamento e identificação de referências.
O layout.py é responsável por definir a estrutura visual da página, nomeadamente o cabeçalho, as secções divididas que organizam o mapa 2D, a vista 3D, o radar, os gráficos temporais daprofundidade e velocidade e a barra inferior de reprodução, onde é possível aumentar e diminuir o ritmo de reprodução. O callbacks.py contém todos os callbacks do Dash que gerem o dinamismo da dashboard. É aqui que o movimento dos robôs é sincronizado com a barra de reprodução, calculadas as trajetórias, atualizados os gráficos do Plotly e gerada a perspetiva 3D.

### 2.1. Transformações de coordenadas e matrizes de rotação

O dashboard usa duas matrizes de rotação, ambas implementadas com NumPy no ficheiro `callbacks.py`.

**Orientação do robô** Cada registo dos logs inclui a orientação do robô como quaternião unitário $q = (q_x, q_y, q_z, q_w)$. Para desenhar os eixos do robô na bússola 3D, o quaternião é convertido na matriz de rotação

$$
R(q)=\begin{bmatrix}
1-2(q_y^2+q_z^2) & 2(q_xq_y-q_zq_w) & 2(q_xq_z+q_yq_w)\\
2(q_xq_y+q_zq_w) & 1-2(q_x^2+q_z^2) & 2(q_yq_z-q_xq_w)\\
2(q_xq_z-q_yq_w) & 2(q_yq_z+q_xq_w) & 1-2(q_x^2+q_y^2)
\end{bmatrix}
$$

As colunas de $R$ são os eixos $X$, $Y$ e $Z$ do robô expressos no referencial do mundo, e são esses vetores que a bússola desenha.

**Posição relativa entre robôs** As posições dos logs estão em coordenadas geodésicas $(\varphi, \lambda, h)$. Para calcular distâncias entre robôs, cada posição é convertida para ECEF

$$
\begin{aligned}
X &= (N+h)\cos\varphi\cos\lambda, \quad Y = (N+h)\cos\varphi\sin\lambda, \quad Z = \big(N(1-e^2)+h\big)\sin\varphi,\\
N &= \frac{a}{\sqrt{1-e^2\sin^2\varphi}},
\end{aligned}
$$

e a diferença para a origem $\Delta = \mathbf{p}_{ECEF}-\mathbf{p}_{ECEF,0}$ é rodada para o referencial local NED, centrado em $(\varphi_0,\lambda_0,h_0)$:

$$
\begin{bmatrix}N\\E\\D\end{bmatrix}=
\begin{bmatrix}
-\sin\varphi_0\cos\lambda_0 & -\sin\varphi_0\sin\lambda_0 & \cos\varphi_0\\
-\sin\lambda_0 & \cos\lambda_0 & 0\\
-\cos\varphi_0\cos\lambda_0 & -\cos\varphi_0\sin\lambda_0 & -\sin\varphi_0
\end{bmatrix}\Delta
$$

Esta matriz é ortogonal, com determinante $+1$, e resulta da composição de duas rotações elementares: uma em torno de $Z$ pela longitude $\lambda_0$ e outra em torno de $Y$ pelo ângulo $-(\varphi_0+90^\circ)$. A partir de $(N,E,D)$ calculam-se a distância horizontal $\sqrt{N^2+E^2}$, a distância 3D $\sqrt{N^2+E^2+D^2}$ e o azimute $\operatorname{atan2}(E,N)$, medido a partir do norte no sentido horário, como no sistema AER apresentado nas aulas.

### 3. Aspeto visual da dashboard desenvolvida


A arquitetura final desta dashboard, abaixo ilustrada, demonstra o alcançe dos objetivos propostos no início do desenvolvimento da dashboard. Desta forma, a dashboard permite a visualização individual e coletiva das missões, plotando as mesmas numa vista 2D e 3D, para além de ser possível, através do OpenSeaMap, visualizar o local exato onde foram realizadas as missões. 

<div align="center">
    <img src="imagens/mapa_2d.png" alt= "Mapa 2D das Trajetórias" width="800px">
    <br>
    <em>Figura 1: Vista superior (Mapa 2D) com as trajetórias sobrepostas no OpenSeaMap.</em>
</div>

<br>

### 3.1. Análise Tridimensional e Radar de Distâncias

Esta figura combina a análise de profundidade com a proximidade relativa. A **Vista 3D** permite explorar a trajetória no espaço tridimensional ($x, y, z$). Em simultâneo, o **Radar** monitoriza a distância horizontal e azimute entre os robôs, caso estejam a ser ilustradas as três trajetórias.

<div align="center">
    <img src="imagens/mapa_3d.png" alt="Vista 3D e Radar" width="800px">
    <br>
    <em>Figura 2: Perspetiva 3D das rotas integrando a análise de profundidade (eixo Z), juntamente com a interface do Radar de proximidade.</em>
</div>

<br>

### 3.2. Radar e Gráficos Temporais de profundidade e velocidade

Uma informação que consta nos ficheiros CSV e que também consideramos importante disponibilizar na dashboard é a informação de telemetria ao longo do tempo decorrido na missão. O Radar está em destaque para monitorização de proximidade, enquanto os gráficos inferiores mostram a evolução temporal da **Profundidade** e da **Velocidade Linear** de cada robô, permitindo identificar padrões de comportamento ou possíveis anomalias durante a missão.

<div align="center">
    <img src="imagens/radar_telemetria.png" alt="Radar e Gráficos Temporais" width="800px">
    <br>
    <em>Figura 3: Detalhe do Radar de distâncias e gráficos de telemetria temporal (Profundidade e Velocidade) sincronizados com a barra de reprodução.</em>
</div>



## 5. Conclusões e Trabalho Futuro

O desenvolvimento desta dashboard provou ser uma solução para a visualização e análise integrada de logs de navegação de robôs autónomos em ambiente subaquático. Este projeto permitiu ainda abrir horizontes nomeadamente nas capacidades da linguagem python e no desenvolvimento que as suas bibliotecas permitem desenvolver. Sem dúvida que este foi uma vantagem a realçar para o grupo, apesar da dificuldade sentida no início do desenvolvimento do projeto, dada a reduzida experiência com Python. Para além disso, dado os objetivos propostos inicialmente, tivemos que transportar conhecimentos teóricos para o desenvolvimento desta dashboard, nomeadamente rotações de matrizes, transformações de coordenadas.

A integração bem-sucedida de mapas geográficos em 2D com suporte OpenSeaMap, de uma vista tridimensional otimizada com cota de profundidade e nível do mar, de um radar de proximidade baseado em conversão de coordenadas geodésicas para NED, e de gráficos temporais sincronizados permitiu ao grupo perceber e monitorizar em detalhe o comportamento e a trajetória destes veículos, pelo que, em caso de necessidade de avaliar uma determinada missão, a utilização desta dashboard passa por uma resposta a essa necessidade. 

Como perspetivas de evolução e melhoria contínua para trabalhos futuros, consideramos importante a implementação de um alerta caso os veículos possam colidir ou se encontrem a uma distância de risco. Outra oportunidade de melhoria seria dotar a dashboard com capacidade para operar em tempo real, ou seja, através de uma ligação WebSocket que permita receber mensagens e finalmente atualizar a dashboard, permitindo assim utilizar a mesma não só com ficheiros CSV.

 